# Assignment 2 DSC 102 FA23

## Introduction

In this assignment we will conduct data engineering for the Amazon dataset. It is divided into 2 parts. The extracted features in Part 1 will be used for the Part 2 of assignment, where you train a model (or models) to predict user ratings for a product.

We will be using Apache Spark for this assignment. The default Spark API will be DataFrame, as it is now the recommended choice over the RDD API. That being said, please feel free to switch back to the RDD API if you see it as a better fit for the task. We provide you an option to request RDD format to start with. Also you can switch between DataFrame and RDD in your solution. 

Another newer API is Koalas, which is also avaliable. However, it has constraints and is not applicable to most tasks. Refer to the PA statement for detail.

### Set the following parameters

In [1]:
PID = 'A17367248' # your pid, for instance: 'a43223333'
INPUT_FORMAT = 'dataframe' # choose a format of your input data, valid options: 'dataframe', 'rdd', 'koalas'

In [2]:
# Boiler plates, do NOT modify
%load_ext autoreload
%autoreload 2
import os
import getpass
from pyspark.sql import SparkSession
from utilities import SEED
from utilities import PA2Test
from utilities import PA2Data
from utilities import data_cat
from pa2_main import PA2Executor
import time
if INPUT_FORMAT == 'dataframe':
    import pyspark.ml as M
    import pyspark.sql.functions as F
    import pyspark.sql.types as T
if INPUT_FORMAT == 'koalas':
    import databricks.koalas as ks
elif INPUT_FORMAT == 'rdd':
    import pyspark.mllib as M
    from pyspark.mllib.feature import Word2Vec
    from pyspark.mllib.linalg import Vectors
    from pyspark.mllib.linalg.distributed import RowMatrix

os.environ['PYSPARK_SUBMIT_ARGS'] = '--py-files utilities.py,assignment2.py \
--deploy-mode client \
pyspark-shell'

class args:
    review_filename = data_cat.review_filename
    product_filename = data_cat.product_filename
    product_processed_filename = data_cat.product_processed_filename
    ml_features_train_filename = data_cat.ml_features_train_filename
    ml_features_test_filename = data_cat.ml_features_test_filename
    output_root = '/home/{}/{}-pa2/test_results'.format(getpass.getuser(), PID)
    test_results_root = data_cat.test_results_root
    pid = PID

pa2 = PA2Executor(args, input_format=INPUT_FORMAT)
data_io = pa2.data_io
data_dict = pa2.data_dict
begin = time.time()


25/06/05 02:16:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/bitnami/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Loading datasets ...Done


In [3]:
# Import your own dependencies
import pandas as pd
import numpy as np
import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql.types import FloatType
from pyspark.sql.functions import col, explode, mean, variance, size, count, when
from pyspark.sql import functions as F
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
#-----------------------------

# Part 1: Feature Engineering

In [4]:
# Bring the part_1 datasets to memory and de-cache part_2 datasets. 
# Execute this once before you start working on this Part
data_dict, _ = data_io.cache_switch(data_dict, 'part_1')

# Task0: warm up 
This task is provided for you to get familiar with Spark API. We will use the dataframe API to demonstrate. Solution is given to you and this task won't be graded.

Refer to https://spark.apache.org/docs/latest/api/python/pyspark.sql.html for API guide.

The task is to implement the function below. Given the ```product_data``` table:
1. Take and print five rows.

1. Select only the ```asin``` column, then print five rows of it.

1. Select the row where ```asin = B00I8KEOTM``` and print it.

1. Count the total number of rows.

1. Calculate the mean ```price```.

1. You need to conduct the above operations, then extract some statistics out of the generated columns. You need to put the statistics in a python dictionary named ```res```. The description and schema of it are as follows:
    ```
    res
     | -- count_total: int -- count of total rows of the entire table after your operations
     | -- mean_price: float -- mean value of column price
    ```

In [5]:
def task_0(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    overall_column = 'overall'
    # Outputs:
    mean_rating_column = 'meanRating'
    count_rating_column = 'countRating'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------

    product_data.show(5)
    product_data[['asin']].show(5)
    product_data.where(F.col('asin') == 'B00I8KEOTM').show()
    count_rows = product_data.count()
    mean_price = product_data.select(F.avg(F.col('price'))).head()[0]
    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    # Calculate the values programmatically. Do not change the keys and do not
    # hard-code values in the dict. Your submission will be evaluated with
    # different inputs.
    # Modify the values of the following dictionary accordingly.
    res = {'count_total': None, 'mean_price': None}
    
    # Modify res:

    res['count_total'] = count_rows
    res['mean_price'] = mean_price

    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    return res
    # -------------------------------------------------------------------------

In [6]:
if INPUT_FORMAT == 'dataframe':
    res = task_0(data_io, data_dict['product'])
    pa2.tests.test(res, 'task_0')

+----------+--------------------+--------------------+--------------------+-----+--------------------+
|      asin|           salesRank|          categories|               title|price|             related|
+----------+--------------------+--------------------+--------------------+-----+--------------------+
|B00I8HVV6E|{Home &amp; Kitch...|[[Home & Kitchen,...|Intelligent Desig...|27.99|{also_viewed -> [...|
|B00I8KEOTM|                null|[[Apps for Androi...|                null| null|{also_viewed -> [...|
|B00I8KCW4G|{Clothing -> 2233...|[[Clothing, Shoes...|eShakti Women's P...|41.95|{also_viewed -> [...|
|B00I8JKCQW|{Clothing -> 1405...|[[Clothing, Shoes...|Lady Slimming Mid...| null|{also_viewed -> [...|
|B00I8JKI8E|{Home &amp; Kitch...|[[Clothing, Shoes...|3 Tier Bangle Bra...|24.99|{also_viewed -> [...|
+----------+--------------------+--------------------+--------------------+-----+--------------------+
only showing top 5 rows

+----------+
|      asin|
+----------+
|B00I8HVV

tests for task_0 --------------------------------------------------------------
2/2 passed
-------------------------------------------------------------------------------


# Task1

In [11]:
# %load -s task_1 assignment2.py
def task_1(data_io, review_data, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    overall_column = 'overall'
    # Outputs:
    mean_rating_column = 'meanRating'
    count_rating_column = 'countRating'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    joined = (review_data.select(['asin', 'overall', 'reviewerID'])
        .groupBy('asin')
        .agg({'reviewerID' : 'count', 'overall' : 'mean'}))
    joined_stats = product_data.select(['asin']).join(joined, 'asin', 'left')
    
    # Produces columns 'avg(overall)' and 'count(title)'

    #joined_stats = joined_stats.withColumnsRenamed({: 'mean_rating', 'count(title)':'count_rating'})
    mean_meanRating = joined_stats.select(F.avg(F.col('avg(overall)'))).head()[0]
    #print(mean_meanRating)
    count_total = joined_stats.count()
    variance_meanRating = joined_stats.select(F.variance(F.col('avg(overall)'))).head()[0]
    numNulls_meanRating = count_total - joined_stats.na.drop().count()
    mean_countRating = joined_stats.select(F.avg(F.col('count(reviewerID)'))).head()[0]
    variance_countRating = joined_stats.select(F.variance(F.col('count(reviewerID)'))).head()[0]
    numNulls_countRating = count_total - joined_stats.na.drop().count()
    
    #count_total = len(joined_stats)
    



    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    # Calculate the values programmaticly. Do not change the keys and do not
    # hard-code values in the dict. Your submission will be evaluated with
    # different inputs.
    # Modify the values of the following dictionary accordingly.
    res = {
        'count_total': None,
        'mean_meanRating': None,
        'variance_meanRating': None,
        'numNulls_meanRating': None,
        'mean_countRating': None,
        'variance_countRating': None,
        'numNulls_countRating': None
    }
    # Modify res:
    res['count_total'] = count_total
    res['mean_meanRating'] = mean_meanRating
    res['variance_meanRating'] = variance_meanRating
    res['numNulls_meanRating'] = numNulls_meanRating
    res['mean_countRating'] = mean_countRating
    res['variance_countRating'] = variance_countRating
    res['numNulls_countRating'] = numNulls_countRating
    



    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_1')
    return res
    # -------------------------------------------------------------------------

In [12]:
res = task_1(data_io, data_dict['review'], data_dict['product'])
pa2.tests.test(res, 'task_1')

ERROR:root:KeyboardInterrupt while sending command.============>  (20 + 1) / 21]
Traceback (most recent call last):
  File "/opt/bitnami/python/lib/python3.8/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/bitnami/python/lib/python3.8/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/bitnami/python/lib/python3.8/socket.py", line 669, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 


# Task 2

In [ ]:
# %load -s task_2 assignment2.py
def task_2(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    salesRank_column = 'salesRank'
    categories_column = 'categories'
    asin_column = 'asin'
    # Outputs:
    category_column = 'category'
    bestSalesCategory_column = 'bestSalesCategory'
    bestSalesRank_column = 'bestSalesRank'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------

    categories = product_data.select(
        F.col("categories"),
        F.when(
            (F.col("categories").isNotNull()) &
            (F.size(F.col("categories")) > 0) &
            (F.size(F.col("categories")[0]) > 0) &
            (F.trim(F.col("categories")[0][0]) != ''),
            F.col("categories")[0][0]
        ).otherwise(None)
    )
        
    salesRank_data = product_data.select(['asin', 'salesRank', F.explode(F.col('salesRank'))])
    salesRank_data = salesRank_data.withColumnRenamed('key', 'bestSalesCategory') \
                               .withColumnRenamed('value','bestSalesRank')
    salesRank_data = salesRank_data.drop('salesRank')
    count_total = categories.count()
    mean_bestSalesRank = salesRank_data.select(F.avg(F.col('bestSalesRank'))).head()[0]
    variance_bestSalesRank = salesRank_data.select(F.variance(F.col('bestSalesRank'))).head()[0]
    numNulls_category = count_total - categories.dropna(subset = categories.columns[1]).count()
    countDistinct_category = categories.dropna().select(F.count_distinct(F.col(categories.columns[1]))).head()[0]
    numNulls_bestSalesCategory = count_total - salesRank_data.dropna(subset = 'bestSalesCategory').count() #0 nulls.select('bestSalesRank')[0]
    countDistinct_bestSalesCategory = salesRank_data.select((F.count_distinct(F.col('bestSalesCategory')))).head()[0]
    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': count_total,
        'mean_bestSalesRank':mean_bestSalesRank,
        'variance_bestSalesRank': variance_bestSalesRank,
        'numNulls_category': numNulls_category,
        'countDistinct_category':countDistinct_category,
        'numNulls_bestSalesCategory': numNulls_bestSalesCategory ,
        'countDistinct_bestSalesCategory': countDistinct_bestSalesCategory
    }
    # Modify res:
    res['count_total'] = count_total
    res['mean_bestSalesRank'] = mean_bestSalesRank
    res['variance_bestSalesRank'] = variance_bestSalesRank
    res['numNulls_category'] = numNulls_category
    res['countDistinct_category'] = countDistinct_category
    res['numNulls_bestSalesCategory']= numNulls_bestSalesCategory
    res['countDistinct_bestSalesCategory'] = countDistinct_bestSalesCategory
     

    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_2')
    return res#, salesRank_data
    # -------------------------------------------------------------------------

In [ ]:
res = task_2(data_io, data_dict['product'])
pa2.tests.test(res, 'task_2')

# Task 3





In [ ]:
# %load -s task_3 assignment2.py
def task_3(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    price_column = 'price'
    attribute = 'also_viewed'
    related_column = 'related'
    # Outputs:
    meanPriceAlsoViewed_column = 'meanPriceAlsoViewed'
    countAlsoViewed_column = 'countAlsoViewed'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    
    price_df = product_data.select(asin_column, price_column).where(col(price_column).isNotNull())

    
    also_viewed_df = product_data.select(
        col(asin_column),
        col(f"{related_column}.{attribute}").alias("also_viewed")
    )

    
    count_df = also_viewed_df.withColumn(
        countAlsoViewed_column,
        when(col("also_viewed").isNull() | (size("also_viewed") == 0), None)
        .otherwise(size("also_viewed"))
    ).select(asin_column, countAlsoViewed_column)

    
    exploded = also_viewed_df.where(col("also_viewed").isNotNull()) \
        .withColumn("viewed_asin", explode("also_viewed"))

   
    joined = exploded.join(
        price_df,
        exploded["viewed_asin"] == price_df[asin_column],
        how="left"
    ).select(exploded[asin_column].alias("main_asin"), "price")

   
    filtered = joined.where(col("price").isNotNull())

    
    mean_price_df = filtered.groupBy("main_asin").agg(
        mean("price").alias(meanPriceAlsoViewed_column)
    )


    result_df = product_data.select(asin_column) \
        .join(count_df, on=asin_column, how="left") \
        .join(mean_price_df, product_data[asin_column] == mean_price_df["main_asin"], how="left") \
        .drop("main_asin")


    summary = result_df.selectExpr(
        "COUNT(*) as count_total",
        f"AVG({meanPriceAlsoViewed_column}) as mean_meanPriceAlsoViewed",
        f"VARIANCE({meanPriceAlsoViewed_column}) as variance_meanPriceAlsoViewed",
        f"SUM(CASE WHEN {meanPriceAlsoViewed_column} IS NULL THEN 1 ELSE 0 END) as numNulls_meanPriceAlsoViewed",
        f"AVG({countAlsoViewed_column}) as mean_countAlsoViewed",
        f"VARIANCE({countAlsoViewed_column}) as variance_countAlsoViewed",
        f"SUM(CASE WHEN {countAlsoViewed_column} IS NULL THEN 1 ELSE 0 END) as numNulls_countAlsoViewed"
    ).first().asDict()
    

  
    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'mean_meanPriceAlsoViewed': None,
        'variance_meanPriceAlsoViewed': None,
        'numNulls_meanPriceAlsoViewed': None,
        'mean_countAlsoViewed': None,
        'variance_countAlsoViewed': None,
        'numNulls_countAlsoViewed': None
    }
    
    # -------------------------------------------------------------------------
    res['count_total']: summary['count_total']
    res['mean_meanPriceAlsoViewed'] = summary['mean_meanPriceAlsoViewed']
    res['variance_meanPriceAlsoViewed'] = summary['variance_meanPriceAlsoViewed']
    res['numNulls_meanPriceAlsoViewed'] = summary['numNulls_meanPriceAlsoViewed']
    res['mean_countAlsoViewed'] = summary['mean_countAlsoViewed']
    res['variance_countAlsoViewed'] = summary['variance_countAlsoViewed']
    res['numNulls_countAlsoViewed']: summary['numNulls_countAlsoViewed']


    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_3')
    return res
    # -------------------------------------------------------------------------

In [ ]:
res = task_3(data_io, data_dict['product'])
pa2.tests.test(res, 'task_3')

# Task 4

In [ ]:
# %load -s task_4 assignment2.py
def task_4(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    price_column = 'price'
    title_column = 'title'
    # Outputs:
    meanImputedPrice_column = 'meanImputedPrice'
    medianImputedPrice_column = 'medianImputedPrice'
    unknownImputedTitle_column = 'unknownImputedTitle'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    price_data = product_data.select(['price', 'title'])
    price_data = price_data.withColumn('price', F.col('price').cast(FloatType()))
    
    mean_price = price_data.select(F.avg('price')).collect()[0][0]
    price_data = price_data.withColumn(
        'meanImputedPrice',
        F.when(F.col('price').isNotNull(), F.col('price')).otherwise(F.lit(mean_price))
    )
    
    median_price = price_data.approxQuantile("price", [0.5], 0.01)[0]
    price_data = price_data.withColumn(
        'medianImputedPrice',
        F.when(F.col('price').isNotNull(), F.col('price')).otherwise(F.lit(median_price))
    )
    
    price_data = price_data.withColumn(
        'unknownImputedTitle',
        F.when(F.col('title').isNotNull() & (F.col('title')!=''), F.col('title')).otherwise(F.lit('unknown'))
    )
    
    # Calculating values for res
    count_total = price_data.count()
    mean_meanImputedPrice = price_data.select(F.avg('meanImputedPrice')).collect()[0][0]
    variance_meanImputedPrice = price_data.select(F.variance('meanImputedPrice')).collect()[0][0]
    numNulls_meanImputedPrice = count_total - price_data.select('meanImputedPrice').na.drop().count()
    mean_medianImputedPrice = price_data.select(F.avg('medianImputedPrice')).collect()[0][0]
    variance_medianImputedPrice = price_data.select(F.variance('medianImputedPrice')).collect()[0][0]
    numNulls_medianImputedPrice = count_total - price_data.select('medianImputedPrice').na.drop().count()
    numUnknowns_unknownImputedTitle = price_data.filter(F.col('unknownImputedTitle') == 'unknown').count()




    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'mean_meanImputedPrice': None,
        'variance_meanImputedPrice': None,
        'numNulls_meanImputedPrice': None,
        'mean_medianImputedPrice': None,
        'variance_medianImputedPrice': None,
        'numNulls_medianImputedPrice': None,
        'numUnknowns_unknownImputedTitle': None
    }
    
    
    # Modify res:
    res['count_total'] = count_total
    res['mean_meanImputedPrice' = mean_meanImputedPrice
    res['variance_meanImputedPrice'] = variance_meanImputedPrice
    res['numNulls_meanImputedPrice'] = numNulls_meanImputedPrice
    res['mean_medianImputedPrice'] = mean_medianImputedPrice
    res['variance_medianImputedPrice'] = variance_medianImputedPrice
    res['numNulls_medianImputedPrice'] = numNulls_medianImputedPrice
    res['numUnknowns_unknownImputedTitle'] = numUnknowns_unknownImputedTitle



    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_4')
    return res
    # -------------------------------------------------------------------------


In [ ]:
res = task_4(data_io, data_dict['product'])
pa2.tests.test(res, 'task_4')

# Task 5

In [ ]:
# %load -s task_5 assignment2.py

def task_5(data_io, product_processed_data, word_0, word_1, word_2):
    # -----------------------------Column names--------------------------------
    # Inputs:
    title_column = 'title'
    # Outputs:
    titleArray_column = 'titleArray'
    titleVector_column = 'titleVector'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    title_data = product_processed_data.select('title')
    product_processed_data_output = title_data.withColumn('titleArray', F.split(F.lower(F.col('title')), " "))
    word2Vec = M.feature.Word2Vec(
        vectorSize=16, 
        minCount=100, 
        seed=SEED, 
        inputCol="titleArray", 
        outputCol="titleVector", 
        numPartitions=4
    )
    model = word2Vec.fit(product_processed_data_output)
    product_processed_data_output = model.transform(product_processed_data_output)

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'size_vocabulary': None,
        'word_0_synonyms': [(None, None), ],
        'word_1_synonyms': [(None, None), ],
        'word_2_synonyms': [(None, None), ]
    }
    # Modify res:
    res['count_total'] = product_processed_data_output.count()
    res['size_vocabulary'] = model.getVectors().count()
    for name, word in zip(
        ['word_0_synonyms', 'word_1_synonyms', 'word_2_synonyms'],
        [word_0, word_1, word_2]
    ):
        res[name] = model.findSynonymsArray(word, 10)
    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_5')
    return res
    # -------------------------------------------------------------------------


In [ ]:
res = task_5(data_io, data_dict['product_processed'], 'piano', 'rice', 'laptop')
pa2.tests.test(res, 'task_5')

# Task 6

In [ ]:
# %load -s task_6 assignment2.py
def task_6(data_io, product_processed_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    category_column = 'category'
    # Outputs:
    categoryIndex_column = 'categoryIndex'
    categoryOneHot_column = 'categoryOneHot'
    categoryPCA_column = 'categoryPCA'
    # -------------------------------------------------------------------------    

    # ---------------------- Your implementation begins------------------------
    categories = product_processed_data.select(F.col(category_column))
    stringIndexer = M.feature.StringIndexer(
        inputCol=category_column,
        outputCol=categoryIndex_column,
    )
    cat_indexes = stringIndexer.fit(categories)
    indexed = cat_indexes.transform(categories)
    
    onehot = M.feature.OneHotEncoder(
        inputCol=categoryIndex_column,
        outputCol=categoryOneHot_column,
        dropLast=False
    )
    onehot_model = onehot.fit(indexed)
    onehot_data = onehot_model.transform(indexed)

    pca = M.feature.PCA(
        k=15,
        inputCol=categoryOneHot_column,
        outputCol = categoryPCA_column
    )
    pca_model = pca.fit(onehot_data)
    pca_data = pca_model.transform(onehot_data)
    
    count_total = pca_data.count()
    
    summarizer = M.stat.Summarizer()
    mean_row_onehot = pca_data.select(
        summarizer.mean(F.col(categoryOneHot_column)).alias("meanVector_categoryOneHot")
    ).first()
    meanVector_categoryOneHot = mean_row_onehot["meanVector_categoryOneHot"]
    
    mean_row_pca = pca_data.select(
        summarizer.mean(F.col(categoryPCA_column)).alias("meanVector_categoryPCA")
    ).first()
    meanVector_PCA = mean_row_pca["meanVector_categoryPCA"]

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': None,
        'meanVector_categoryOneHot': [None, ],
        'meanVector_categoryPCA': [None, ]
    }
    # Modify res:
    res['count_total'] = count_total
    res['meanVector_categoryOneHot'] = meanVector_categoryOneHot
    res['meanVector_categoryPCA'] = meanVector_PCA



    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_6')
    return res
    # -------------------------------------------------------------------------


In [ ]:
res = task_6(data_io, data_dict['product_processed'])
pa2.tests.test(res, 'task_6')

In [ ]:
print ("End to end time: {}".format(time.time()-begin))

# Part 2: Model Selection

In [ ]:
# Bring the part_2 datasets to memory and de-cache part_1 datasets.
# Execute this once before you start working on this Part
data_dict, _ = data_io.cache_switch(data_dict, 'part_2')

# Task 7

In [13]:
def task_7(data_io, train_data, test_data):
    # ---------------------- Your implementation begins------------------------

  
    
    label_col = 'overall' if 'overall' in train_data.columns else 'label'

    
    train_vector = train_data.select("features", label_col)
    test_vector = test_data.select("features", label_col)

    
    dt = DecisionTreeRegressor(featuresCol="features", labelCol=label_col, maxDepth=5)
    model = dt.fit(train_vector)

    
    predictions = model.transform(test_vector)

   
    evaluator = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
    test_rmse = evaluator.evaluate(predictions)

    
    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'test_rmse': test_rmse
    }
    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_7')
    return res
    # -------------------------------------------------------------------------

In [14]:
res = task_7(data_io, data_dict['ml_features_train'], data_dict['ml_features_test'])
pa2.tests.test(res, 'task_7')

Exception in thread "serve-DataFrame" java.net.SocketTimeoutException: Accept timed out
	at java.net.PlainSocketImpl.socketAccept(Native Method)
	at java.net.AbstractPlainSocketImpl.accept(AbstractPlainSocketImpl.java:409)
	at java.net.ServerSocket.implAccept(ServerSocket.java:560)
	at java.net.ServerSocket.accept(ServerSocket.java:528)
	at org.apache.spark.security.SocketAuthServer$$anon$1.run(SocketAuthServer.scala:64)


tests for task_7 --------------------------------------------------------------
Test 1/1 : test_rmse ... Pass
1/1 passed
-------------------------------------------------------------------------------


True

# Task 8

In [15]:
def task_8(data_io, train_data, test_data):
    
    # ---------------------- Your implementation begins------------------------
    
    label_col = 'overall' if 'overall' in train_data.columns else 'label'
    
    train_df, test_df = train_data.randomSplit([0.75,0.25], seed=SEED)
    train_vector = train_df.select("features", label_col)
    valid_vector = test_df.select("features", label_col)
    test_vector = test_data.select("features", label_col)
    
    dt_5 = DecisionTreeRegressor(featuresCol="features", labelCol=label_col, maxDepth=5)
    model_5 = dt_5.fit(train_vector)
    predictions_5 = model_5.transform(valid_vector)
    evaluator_5 = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
    valid_rmse_depth_5 = evaluator.evaluate(predictions_5)
    
    dt_7 = DecisionTreeRegressor(featuresCol="features", labelCol=label_col, maxDepth=7)
    model_7 = dt_7.fit(train_vector)
    predictions_7 = model_7.transform(valid_vector)
    evaluator_7 = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
    valid_rmse_depth_7 = evaluator.evaluate(predictions_7)
    
    dt_9 = DecisionTreeRegressor(featuresCol="features", labelCol=label_col, maxDepth=7)
    model_9 = dt_9.fit(train_vector)
    predictions_9 = model_9.transform(valid_vector)
    evaluator_9 = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
    valid_rmse_depth_9 = evaluator.evaluate(predictions_9)
    
    dt_12 = DecisionTreeRegressor(featuresCol="features", labelCol=label_col, maxDepth=7)
    model_12 = dt_12.fit(train_vector)
    predictions_12 = model_12.transform(valid_vector)
    evaluator_12 = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
    valid_rmse_depth_12 = evaluator.evaluate(predictions_12)
    
    min_rmse = min(valid_rmse_depth_5, valid_rmse_depth_7, valid_rmse_depth_9, valid_rmse_depth_12)
    if min_rmse == valid_rmse_depth_5:
        pred = model_5.transform(test_vector)
    elif min_rmse == valid_rmse_depth_7:
        pred = model_7.transform(test_vector)
    elif min_rmse == valid_rmse_depth_9:
        pred = model_9.transform(test_vector)
    else:
        pred = model_12.transform(test_vector)
    test_rmse = evaluator.evaluate(pred)
    
    
    # -------------------------------------------------------------------------
    
    
    # ---------------------- Put results in res dict --------------------------
    res = {
        'test_rmse': None,
        'valid_rmse_depth_5': None,
        'valid_rmse_depth_7': None,
        'valid_rmse_depth_9': None,
        'valid_rmse_depth_12': None,
    }
    # Modify res:
    res['test_rmse'] = test_rmse
    res['valid_rmse_depth_5'] = valid_rmse_depth_5
    res['valid_rmse_depth_7'] = valid_rmse_depth_7
    res['valid_rmse_depth_9'] = valid_rmse_depth_9
    res['valid_rmse_depth_12'] = valid_rmse_depth_12

    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_8')
    return res
    # -------------------------------------------------------------------------

In [16]:
res = task_8(data_io, data_dict['ml_features_train'], data_dict['ml_features_test'])
pa2.tests.test(res, 'task_8')

NameError: name 'evaluator' is not defined

In [ ]:
print ("End to end time: {}".format(time.time()-begin))